# Redback Simulations

Use [redback](https://redback.readthedocs.io/en/latest/simulation.html#examples) to simulate TDEs and other transients at different redshifts.

### summary of selected simulated objects:

TDEs
- ZTF17aabvong with redback TDE_analytical model (phenomological)
- tde_analytic model with hand set parameters (model based)
- tde_fallback (similar to mosfit) model with hand set parameters (model based)

SN
- ZTF17aabvong with salt2 model (phenomological)

In [ ]:
import warnings
# Suppress SyntaxWarnings from invalid escape sequences in redback/bilby source files
warnings.filterwarnings('ignore', category=SyntaxWarning)
# Suppress optional-dependency warnings from redback (lalsimulation, swifttools)
warnings.filterwarnings('ignore', message='.*lalsimulation.*')
warnings.filterwarnings('ignore', message='.*swifttools.*')

import numpy as np
import pandas as pd
import os
import bilby
import redback  
from astropy.cosmology import Planck18 as cosmo                                                                                                                   
import matplotlib.pyplot as plt
import matplotlib              
from matplotlib import rcParams
rcParams["font.family"] = "Liberation Serif"
matplotlib.rcParams['text.usetex'] = False

In [ ]:
REDBACK_OUTDIR = '../../data/redback_sims/redback_inference'  # save/load all inference results here

## Fetch real object photometry

redback has [built in tools to get data](https://redback.readthedocs.io/en/latest/getting_data.html) from open access, otter catalogs and fink / lasair brokers

In [ ]:
# which real tdes can we retrieve from open access catalog

oac_path = os.path.join(os.path.dirname(redback.__file__), 'tables', 'OAC_metadata.csv')                                                                          
df = pd.read_csv(oac_path, low_memory=False, on_bad_lines='skip')
tdes = df[df['claimedtype'].str.contains('TDE|tidal', case=False, na=False)]                                                                                      
                                                                                                                                                                
print(f"Total TDEs in redback OAC catalog: {len(tdes)}")                                                                                                          
#print(tdes['name'].tolist())

In [ ]:
# search for certain characteristics ie bright                                                                                                                                                
                                                                                                                                                                  
oac_path = os.path.join(os.path.dirname(redback.__file__), 'tables', 'OAC_metadata.csv')                                                                          
df = pd.read_csv(oac_path, low_memory=False, on_bad_lines='skip')
tdes = df[df['claimedtype'].str.contains('TDE|tidal', case=False, na=False)].copy()                                                                               
tdes['maxabsmag'] = pd.to_numeric(tdes['maxabsmag'], errors='coerce')                                                                                             
                                                                                                                                                                
# print brightest                                                                                                                            
brightest = tdes[['name', 'maxabsmag']].dropna().sort_values('maxabsmag').head(10)                                                                                 
print("Brightest TDEs by peak absolute magnitude:")                                                                                                             
print(brightest.to_string(index=False))                                                                                                                           
                                                                                                                                                                
# histogram                                                                                                                                                       
fig, ax = plt.subplots(figsize=(8, 5))                                                                                                                          
tdes['maxabsmag'].dropna().plot.hist(bins=20, ax=ax, edgecolor='black')                                                                                           
ax.set_xlabel('Peak Absolute Magnitude', fontsize=18)
ax.set_ylabel('Count', fontsize=18)                                                                                                                                            
ax.set_title('Distribution of TDE Peak Absolute Magnitudes (redback OAC catalog)', fontsize=22)                                                                              
ax.invert_xaxis()  # brighter = more negative, so flip axis                                                                                                       
plt.tight_layout()                                                                                                                                                
plt.show()  

In [ ]:
# instead of filtering for OAC TDEs, just check brightest objects in general

oac_path = os.path.join(os.path.dirname(redback.__file__), 'tables', 'OAC_metadata.csv')                                                                          
df = pd.read_csv(oac_path, low_memory=False, on_bad_lines='skip')
df['maxabsmag'] = pd.to_numeric(df['maxabsmag'], errors='coerce')
brightest = df.dropna(subset=['maxabsmag']).sort_values('maxabsmag').head(15)
print(brightest[['name', 'maxabsmag', 'claimedtype', 'redshift']].to_string())                                                                                   

In [ ]:
# we can also use Otter to query for objects

import otter
import numpy as np

db = otter.Otter(datadir='../../data/redback_sims')

# query for confirmed TDEs with optical photometry in one step
all_transients = db.query(classification='TDE', unambiguous=True,)# has_uvoir_phot=True)

tdes_with_z = []
for t in all_transients:
    try:
        z = t.get_redshift()
    except (KeyError, TypeError):
        continue
    if z is None:
        continue

    z_val = float(np.median([v for v in (z if isinstance(z, (list, np.ndarray)) else [z]) if v is not None]))
    name = t.get('name', {}).get('default_name', str(t))
    tdes_with_z.append((name, z_val))

tdes_with_z.sort(key=lambda x: x[1], reverse=True)

print(f"Found {len(tdes_with_z)} confirmed TDEs\n")
print(f"{'Rank':<6} {'Name':<25} {'Redshift'}")
print("-" * 45)
for i, (name, z) in enumerate(tdes_with_z[:15], 1):
    print(f"{i:<6} {name:<25} {z:.4f}")

In [ ]:
# check for a specific transient in the results by name

target_object = '2019cmw'

for t in all_transients:
    name = t.get('name', {}).get('default_name', str(t))
    if target_object in name:
        print(f"Found target object: {name}")
        break

### now load photometry into redback structure for selected real objects

In [ ]:
import importlib
import thor.utils.redback_sims
importlib.reload(thor.utils.redback_sims)

In [ ]:
# Load with name, for example TDES from otter: AT2019cmw, AT2018iih (z=0.212), AT2020ysg, AT2020acka
name = 'ZTF17aabvong'
source = 'fink_sn'

from thor.utils.redback_sims import load_data_redback_object, create_redback_transient
data = load_data_redback_object(object_name=name, data_source=source, outdir=REDBACK_OUTDIR)

In [ ]:
# create redback transient object from loaded data
exclude_bands = ['W2', 'M2', 'W1', 'U'] # set exclude_bands = None  # utilize all bands available

transient_object = create_redback_transient(
    object_name=name,
    data_source=source,
    data=data,
    data_mode='flux_density',
    exclude_bands=exclude_bands,
    outdir=REDBACK_OUTDIR
)

In [ ]:
# plot

# the black points are bands available that havent been set as active
transient_object.plot_data(show=False, title=f"{name} photometry")

# to plot a single band per subplot, use the following:
#fig, axes = plt.subplots(1,3, sharex=True, sharey=True, figsize=(12, 5))
#transient_object.plot_multiband(figure=fig, axes=axes, filters=["g", "r", "UVW1"], show=False)

Usage warning: 

- check if data are in AB or Vega system (redback assumes AB, throws away explicity flagged Vega from processed file)

- check if extinction correction was applied to OAC data (redback assumes no extinction correction applied)

- check filter names are correct. EX: 'r' is treated as SDSS r but may be referring to R band or ZTF/LSST r band which have different filter responses


# fit model based on real photometry

### [Redback Models](https://redback.readthedocs.io/en/latest/models.html)

- tde model options:
- Simple analytic fallback
- Cooling envelope (Metzger 2022)
- Gaussian rise + cooling envelope (Sarin and Metzger 2024)
- Broken power law + cooling envelope (Sarin and Metzger 2024)
- Fitted
- TDEMass
- Mosfit TDE

In [ ]:
# print all the priors for a given model

priors = redback.priors.get_priors(model='tde_analytical')
priors

In [ ]:
# fit SNIa model to the transient data (ZTF17aabvong, from BTS explorer, z=0.034) 
# use SALT2 model from SNCOSMO for best performance up to cosmological redshifts
# depending on priors etc, this can take 20+ minutes to run.

import bilby                                                                                                                                                              
import numpy as np
import thor.utils.redback_sims
import importlib
importlib.reload(thor.utils.redback_sims)
from thor.utils.redback_sims import fit_redback_model

# Find the approximate MJD peak from your data to set a sensible prior                                                                                                      
t_data = transient_object.x  # MJD times                                                                                                                                  
t_min, t_max = t_data.min(), t_data.max()                                                                                                                                   
t_mid = (t_min + t_max) / 2
print(f"Data MJD range: {t_min:.1f} – {t_max:.1f}, midpoint: {t_mid:.1f}")                                                                                                  
                                                                                                                                                                        
prior_overrides = {                                                                                                                                                         
    'redshift': 0.034,                                                                                                                                                    
    # peak_time default is Uniform(10, 20)
    'peak_time': bilby.core.prior.Uniform(
        minimum=t_min - 20,                                                                                                                                                 
        maximum=t_max + 20,
        name='peak_time',                                                                                                                                                   
        latex_label=r'$t_\mathrm{peak}$ (MJD)',                                                                                                                           
    ),                                                                                                                                                                      
    # t0 is a small time offset (days); keep it tight around 0
    't0': 0,                                                                                                                                                                    
    # x1: SALT2 stretch — standard cosmological prior                                                                                                                       
    'x1': bilby.core.prior.Uniform(minimum=-3, maximum=3, name='x1', latex_label='$x_1$'),                                                                                  
    # c: SALT2 colour — standard prior                                                                                                                                      
    'c': bilby.core.prior.Uniform(minimum=-0.3, maximum=0.3, name='c', latex_label='$c$'),                                                                                  
}                                                                                                                                                                           
                                                                                                                                                                        
result = fit_redback_model(                                                                                                                                                 
    transient_object=transient_object,                                                                                                                                    
    object_name=name,                                                                                                                                                       
    model='salt2',
    nlive=500,                                                                                                                                                             
    prior_overrides=prior_overrides,                                                                                                                                      
    multiband_filters=['ztfg', 'ztfr'],                                                                                                                                     
    outdir_root=REDBACK_OUTDIR,                                                                                                                                             
)    

In [ ]:
# Fit tde_analytical model for known object. 
# depending on priors etc, this can take 20+ minutes to run.

import bilby
import importlib
import thor.utils.redback_sims
importlib.reload(thor.utils.redback_sims)
from thor.utils.redback_sims import fit_redback_model

prior_overrides = {
    'redshift': 0.4030,
    # temperature prior based on TDE blackbodies at peak
    'temperature_0': bilby.core.prior.Uniform(
        minimum=10000, maximum=60000, name='temperature_0', latex_label='$T_0$ (K)'),
    # radius prior around physically expected TDE photosphere size
    'radius_0': bilby.core.prior.LogUniform(
        minimum=1e13, maximum=1e16, name='radius_0', latex_label='$R_0$ (cm)'),
}

result = fit_redback_model(
    transient_object=transient_object,
    object_name=name,
    model='tde_analytical',
    nlive=500,
    prior_overrides=prior_overrides,
    multiband_filters=['ztfg', 'ztfr'],
    outdir_root=REDBACK_OUTDIR,
)

In [ ]:
# try cooling envelope model

sampler = 'dynesty'
model = 'cooling_envelope'
nlive=500
outdir = f'{REDBACK_OUTDIR}/{name}/{model}'
os.makedirs(outdir, exist_ok=True)

# # By default all data is used, select bands for speed or if you only trust some of the data/physics.
# bands = ["u", "r"]
# transient_object.active_bands = bands

# start with default priors
priors = redback.priors.get_priors(model=model)
# Fix priors, ie if known through the metadata that was downloaded alongside the data
priors['redshift'] = 0.4030
# free up the physically interesting parameters (all fixed by default)                                                                                                      
priors['mbh_6'] = bilby.core.prior.LogUniform(                                                                                                                              
    minimum=0.1, maximum=100, name='mbh_6', latex_label='$M_{\\rm BH}/10^6 M_\\odot$')                                                                                      
priors['eta'] = bilby.core.prior.Uniform(                                                                                                                                   
    minimum=0.01, maximum=0.4, name='eta', latex_label='$\\eta$')                                                                                                           
priors['alpha'] = bilby.core.prior.Uniform(                                                                                                                                 
    minimum=0.01, maximum=0.4, name='alpha', latex_label='$\\alpha$')                                                                                                       
priors['beta'] = bilby.core.prior.Uniform(                                                                                                                                  
    minimum=1.0, maximum=6.0, name='beta', latex_label='$\\beta$')

model_kwargs = dict(frequency=transient_object.filtered_frequencies, output_format='flux_density')

# returns a tde result object
result = redback.fit_model(transient=transient_object, model=model, sampler=sampler,
                           model_kwargs=model_kwargs, prior=priors, sample='rslice', nlive=nlive, npool=4, 
                           resume=True, outdir=outdir, label=f"{name}_nlive{str(nlive)}",
                           plot=False)

# result.transient reconstructs from metadata with wrong path, use transient_object directly
model_func = redback.model_library.all_models_dict[model]

# plot results
result.plot_corner(save=False, show=True)
plt.show()
plt.close('all')
transient_object.plot_lightcurve(
    model=model_func,
    posterior=result.posterior,
    model_kwargs=result.model_kwargs,
    random_models=100,
    save=False,
    show=True,
)
transient_object.plot_multiband_lightcurve(
    model=model_func,
    posterior=result.posterior,
    model_kwargs=result.model_kwargs,
    random_models=100,
    filters=["ztfg", "ztfr"],
    save=False,
    show=True,
)

In [ ]:
# Experiment with fit: afterglow SED + blackbody via afterglow_and_optical combined model

# afterglow_and_optical routes all flat sampled kwargs to both sub-models via **model_kwargs.update(),
# so each sub-model picks up only the parameters it needs — no custom wrapper required.

sampler = 'dynesty'
model = 'afterglow_and_optical'
base_afterglow = 'tophat'
base_optical = 'evolving_blackbody'
nlive = 1000
outdir = f'{REDBACK_OUTDIR}/{name}/{model}_{base_afterglow}'
os.makedirs(outdir, exist_ok=True)

# combined flat priors: tophat afterglow + evolving blackbody
priors = redback.priors.get_priors(model=base_afterglow)
priors.update(redback.priors.get_priors(model=base_optical))

# add host extinction (required by afterglow_and_optical)
priors['av'] = bilby.core.prior.LogUniform(minimum=1e-3, maximum=3.0, name='av', latex_label='$A_V$')

# fix redshift; tighten temperature and radius for TDE-like conditions
priors['redshift'] = 0.4030
priors['temperature_0'] = bilby.core.prior.Uniform(
    minimum=10000, maximum=60000, name='temperature_0', latex_label='$T_0$ (K)')
priors['radius_0'] = bilby.core.prior.LogUniform(
    minimum=1e13, maximum=1e16, name='radius_0', latex_label='$R_0$ (cm)')

# fixed kwargs: sub-model names + frequency + format
# sampled parameters flow through as flat kwargs and get routed to each sub-model automatically
model_kwargs = dict(
    frequency=transient_object.filtered_frequencies,
    output_format='flux_density',
    base_afterglow_model=base_afterglow,
    base_optical_model=base_optical,
    afterglow_kwargs={},
    optical_kwargs={},
)

result = redback.fit_model(transient=transient_object, model=model, sampler=sampler,
                           model_kwargs=model_kwargs, prior=priors, sample='rslice', nlive=nlive, npool=4,
                           resume=True, outdir=outdir, label=f"{name}_nlive{str(nlive)}",
                           plot=False)

model_func = redback.model_library.all_models_dict[model]

result.plot_corner(save=False, show=True)
plt.show()
plt.close('all')

transient_object.plot_lightcurve(
    model=model_func,
    posterior=result.posterior,
    model_kwargs=result.model_kwargs,
    random_models=100,
    save=False,
    show=True,
)
transient_object.plot_multiband_lightcurve(
    model=model_func,
    posterior=result.posterior,
    model_kwargs=result.model_kwargs,
    random_models=100,
    filters=["ztfg", "ztfr"],
    save=False,
    show=True,
)

use generated models to simulate object at different redshifts

In [ ]:
# open a saved model and plot the fit 

from thor.utils.redback_sims import load_redback_result

name = 'ZTF18acecugr'
model= 'tde_analytical'

result, transient = load_redback_result(
    name=name,
    data_source='fink_tde',
    model=model,
    nlive=500,
    exclude_bands=['W2', 'M2', 'W1', 'U'],
    base_dir='../../data/redback_sims/redback_inference',
)

In [ ]:
# load the SNIa example

from thor.utils.redback_sims import load_redback_result

name = 'ZTF17aabvong'
model='salt2'

result, transient = load_redback_result(
    name=name,
    data_source='fink_sn',
    model=model,
    nlive=500,
    exclude_bands=['W2', 'M2', 'W1', 'U'],
    base_dir='../../data/redback_sims/redback_inference',
)

In [ ]:
from thor.utils.redback_sims import plot_lsst_projection
                                                                                                                                                                      
fig, ax = plot_lsst_projection(                                                                                                                                             
    result, transient,                                                                                                                                                    
    name=name,                                                                                                                                                  
    model=model,                                                                                                                                                 
    redshift=None
)    

In [ ]:
# inspect object over redshift 0-3
from thor.utils.redback_sims import plot_redshift_evolution

fig, axes = plot_redshift_evolution(
    result,
    name=name,
    model=model,
    transient=transient,
    redshifts = [(0.1, '-'), (0.2, '--'), (0.3, '-.'), (0.5, ':')]
)

# generate transient from manually set parameters and model (no real data)

In [ ]:
# manually fix parameters for tde_analytical model

from redback.transient_models.tde_models import tde_analytical

z = 0.01
times = np.linspace(0, 300, 500)
band_freqs = {'u': 8.6e14, 'g': 6.38e14, 'r': 4.858e14, 'i': 4.005e14, 'z': 3.3e14, 'y': 3.0e14}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {
    'mej': 1.0, 'vej': 1e4, 'kappa': 0.1, 'kappa_gamma': 10.0,
    'temperature_floor': 5000.0, 'l0': 1e55, 't_0_turn': 50.0
}

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [3, 1]})

for band, freq in band_freqs.items():
    flux = tde_analytical(times, redshift=z, **params,
                        frequency=np.full(len(times), freq),
                        output_format='flux_density') * 1e-26  # erg/s/cm²/Hz
    ax.plot(times, flux, color=band_colors[band], label=band)

ax.legend(fontsize=13)
ax.set_xlabel('Time (days)', size=16)
ax.set_ylabel('Flux density (erg/s/cm²/Hz)', size=16)
ax.set_title(f'Example TDE Light Curve z={z} (Analytical Model)', size=18)

ax_text.axis('off')
param_str = f'redshift = {z}\n' + '\n'.join(f'{k} = {v:.3g}' for k, v in params.items())
ax_text.text(0.1, 0.5, param_str, transform=ax_text.transAxes,
            fontsize=12, verticalalignment='center', family='monospace')

plt.tight_layout()
plt.show()

In [ ]:
# try mosfit model

from redback.transient_models.tde_models import tde_fallback

z=0.01
times = np.linspace(0, 300, 500)
band_freqs = {'u': 8.6e14, 'g': 6.38e14, 'r': 4.858e14, 'i': 4.005e14, 'z': 3.3e14, 'y': 3.0e14}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {
    'mbh6': 1.0,      # BH mass in units of 10^6 solar masses
    'mstar': 1.0,     # disrupted star mass (solar masses)
    'tvisc': 10.0,    # viscous timescale (days)
    'bb': 1.0,        # beta/gamma parameter for disrupted star
    'eta': 0.1,       # BH accretion efficiency
    'leddlimit': 1.0, # Eddington limit factor
    'rph0': 1.0,      # initial photosphere radius
    'lphoto': 1.0,    # photosphere luminosity factor
}

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [3, 1]})

for band, freq in band_freqs.items():
    flux = tde_fallback(times, redshift=z, **params,
                        frequency=np.full(len(times), freq),
                        output_format='flux_density') * 1e-26
    ax.plot(times, flux, color=band_colors[band], label=band)

ax.legend(fontsize=13)
ax.set_xlabel('Time (days)', size=16)
ax.set_ylabel('Flux density (erg/s/cm²/Hz)', size=16)
ax.set_title(f'Example TDE Light Curve z={z} (MOSFiT/Fallback Model)', size=18)

ax_text.axis('off')
param_str = f'redshift = {z}\n' + '\n'.join(f'{k} = {v:.3g}' for k, v in params.items())
ax_text.text(0.1, 0.5, param_str, transform=ax_text.transAxes,
            fontsize=12, verticalalignment='center', family='monospace')

plt.tight_layout()
plt.show()

In [ ]:
# tde_analytical model at z=0.01 and z=2 for comparison

times = np.linspace(0, 300, 500)
band_names = {'u': 'lsstu', 'g': 'lsstg', 'r': 'lsstr', 'i': 'lssti','z': 'lsstz', 'y': 'lssty'}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {                                                                                                                                                        
    'mej': 1.0, 'vej': 1e4, 'kappa': 0.1, 'kappa_gamma': 10.0,                                                                                                  
    'temperature_floor': 5000.0, 'l0': 1e55, 't_0_turn': 50.0                                                                                                     
}                                       
                                                                                                                                                                
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [3, 1]})                                                                    
                                                                                    
for band, band_reg in band_names.items():   
    for z, ls in [(0.01, '-'), (2, '--')]:
        mag = tde_analytical(times, redshift=z, **params,                                                                                                         
                            bands=np.full(len(times), band_reg),                                                                                                 
                            output_format='magnitude')                                                                                                           
        ax.plot(times, mag, color=band_colors[band], linestyle=ls,                                                                                                
                label=f'{band} z={z}') 
                                                                                                                     
                                                                                                                                                                
ax.legend(fontsize=11)                                                                                                                                            
ax.set_xlabel('Time (days)', size=16)                                                                                                                             
ax.invert_yaxis()  
ax.set_ylabel('Apparent magnitude', size=16)   
ax.set_title('Example TDE Light Curve (Analytical Model)', size=18)                                                                                               
                                                                                                                                                                
ax_text.axis('off')
param_str = '\n'.join(f'{k} = {v:.3g}' for k, v in params.items())                                                                                                
ax_text.text(0.1, 0.5, param_str, transform=ax_text.transAxes,
            fontsize=12, verticalalignment='center', family='monospace')                                                                                         
                                                                                                                                                                
plt.tight_layout()                                                                                                                                                
plt.show()

In [ ]:
# mosfit model at z=0.01 and z=2 for comparison

from redback.transient_models.tde_models import tde_fallback

times = np.linspace(0, 300, 500)
band_names = {'u': 'lsstu', 'g': 'lsstg', 'r': 'lsstr', 'i': 'lssti', 'z': 'lsstz', 'y': 'lssty'}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {
    'mbh6': 1.0,
    'mstar': 1.0,
    'tvisc': 10.0,
    'bb': 1.0,
    'eta': 0.1,
    'leddlimit': 1.0,
    'rph0': 1.0,
    'lphoto': 1.0,
}

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [3, 1]})

for band, band_reg in band_names.items():
    for z, ls in [(0.01, '-'), (2, '--')]:
        mag = tde_fallback(times, redshift=z, **params,
                            bands=np.full(len(times), band_reg),
                            output_format='magnitude')
        ax.plot(times, mag, color=band_colors[band], linestyle=ls,
                label=f'{band} z={z}')

ax.legend(fontsize=11)
ax.set_xlabel('Time (days)', size=16)
ax.invert_yaxis()
ax.set_ylabel('Apparent magnitude', size=16)
ax.set_title('Example TDE Light Curve (MOSFiT/Fallback Model)', size=18)

ax_text.axis('off')
param_str = '\n'.join(f'{k} = {v:.3g}' for k, v in params.items())
ax_text.text(0.1, 0.5, param_str, transform=ax_text.transAxes,
            fontsize=12, verticalalignment='center', family='monospace')

plt.tight_layout()
plt.show()

In [ ]:
# tde_analytical model at z=0.1-3, plotting bands separately

times = np.linspace(0, 300, 500)                                                                                                                                  
band_names = {'u': 'lsstu', 'g': 'lsstg', 'r': 'lsstr', 'i': 'lssti', 'z': 'lsstz', 'y': 'lssty'}                                                                 
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}                                                               
params = {                                                                                                                                                        
    'mej': 1.0, 'vej': 1e4, 'kappa': 0.1, 'kappa_gamma': 10.0,
    'temperature_floor': 5000.0, 'l0': 1e55, 't_0_turn': 50.0                                                                                                     
}                                                                                                                                                                 
redshifts = [(0.1, '-'), (0.5, '--'), (1.0, '-.'), (2.0, ':'), (3.0, (0, (3, 1, 1, 1)))]                                                                          
                                                                                                                                                                
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)                                                                                       
                                                                                                                                                                
for ax, (band, band_reg) in zip(axes.flat, band_names.items()):                                                                                                   
    for z, ls in redshifts:                                                                                                                                       
        mag = tde_analytical(times, redshift=z, **params,                                                                                                         
                            bands=np.full(len(times), band_reg),                                                                                                 
                            output_format='magnitude')                                                                                                         
        abs_mag = mag - cosmo.distmod(z).value                                                                                                                    
        ax.plot(times, abs_mag, color=band_colors[band], linestyle=ls, label=f'z={z}')
    ax.set_title(band, fontsize=14, color=band_colors[band])                                                                                                      
    ax.legend(fontsize=9)                                                                                                                                       
                                        
for ax in axes[1]:                                                                                                                                                
    ax.set_xlabel('Time (days)', size=13)                                                                                                                         
for ax in axes[:, 0]:                                                                                                                                             
    ax.set_ylabel('Absolute magnitude', size=13)                                                                                                                  
                                                                                                                                                                
fig.suptitle('TDE Light Curves by Band (Analytical Model)', size=16)
plt.tight_layout()
ax.invert_yaxis()                                                                                                                                                 
plt.show()

In [ ]:
times = np.linspace(0, 1000, 500)
band_names = {'u': 'lsstu', 'g': 'lsstg', 'r': 'lsstr', 'i': 'lssti', 'z': 'lsstz', 'y': 'lssty'}
band_colors = {'u': 'violet', 'g': 'blue', 'r': 'green', 'i': 'orange', 'z': 'red', 'y': 'darkred'}
params = {
    'mbh6': 1.0,
    'mstar': 1.0,
    'tvisc': 10.0,
    'bb': 1.0,
    'eta': 0.1,
    'leddlimit': 1.0,
    'rph0': 1.0,
    'lphoto': 1.0,
}
redshifts = [(0.1, '-'), (0.5, '--'), (1.0, '-.'), (2.0, ':'), (3.0, (0, (3, 1, 1, 1)))]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)

for ax, (band, band_reg) in zip(axes.flat, band_names.items()):
    for z, ls in redshifts:
        mag = tde_fallback(times, redshift=z, **params,
                            bands=np.full(len(times), band_reg),
                            output_format='magnitude')
        abs_mag = mag - cosmo.distmod(z).value
        ax.plot(times, abs_mag, color=band_colors[band], linestyle=ls)
    ax.set_title(band, fontsize=14, color=band_colors[band])

for ax in axes[1]:
    ax.set_xlabel('Time (days)', size=13)
for ax in axes[:, 0]:
    ax.set_ylabel('Absolute magnitude', size=13)

legend_handles = [plt.Line2D([0], [0], color='black', linestyle=ls, label=f'z={z}')
                for z, ls in redshifts]
fig.legend(handles=legend_handles, fontsize=11, loc='upper right',
            bbox_to_anchor=(1.0, 1.0), title='Redshift', title_fontsize=12)

fig.suptitle('TDE Light Curves by Band (MOSFiT/Fallback Model)', size=16)
plt.tight_layout()
ax.invert_yaxis()
plt.show()

In [ ]:
# FIXME

#apply extinction

from redback.transient_models.extinction_models import extinction_with_tde_base_model                                                                                       

# best-fit intrinsic parameters from the TDE fit, with extinction added                                                                                                     
params = {k: v for k, v in result.posterior.median().items()                                                                                                              
        if k not in ['log_likelihood', 'log_prior']}                                                                                                                      
params['av_mw'] = 0.1                                                                                                                                                     
params['av_host'] = 0.0                                                                                                                                                     
                                                                                                                                                                        
times = np.linspace(0, 300, 500)  # observer-frame days
frequencies = tde.filtered_frequencies  # g and i band frequencies

fig, ax = plt.subplots(figsize=(10, 5))                                                                                                                                     

for z in [0.01, 1.0, 1.5, 2.0, 2.5]:                                                                                                                                        
    params['redshift'] = z                                                                                                                                                
    flux = extinction_with_tde_base_model(
        times, **params,
        frequency=np.full(len(times), frequencies[0]),                                                                                                                      
        output_format='flux_density',
        base_model='tde_analytical',                                                                                                                                        
    )                                                                                                                                                                     
    ax.plot(times, flux, label=f'z={z}')

ax.set_xlabel('Observer-frame time (days)')
ax.set_ylabel('Flux density (mJy)')
ax.set_title('g-band flux with MW extinction (av_mw=0.1)')
ax.legend()                                                                                                                                                                 
plt.show()

In [ ]:
# FIXME

# add lyman alpha forest absorption to lightcurves

def igm_transmission(freq_obs, z_source):
    """                                                                                                                                                                     
    IGM HI transmission using Madau (1995) Eq. 15.                                                                                                                        
    freq_obs: observed frequency in Hz (scalar or array)                                                                                                                    
    z_source: source redshift                                                                                                                                               
    Returns: transmission array (0 to 1)
    """                                                                                                                                                                     
    c_AA = 2.998e18  # speed of light in Angstrom/s                                                                                                                       
    wave_obs = c_AA / np.atleast_1d(freq_obs)   # observed wavelength in Angstroms                                                                                          
    wave_emit = wave_obs / (1 + z_source)        # rest-frame wavelength                                                                                                    
                                                                                                                                                                            
    T = np.ones_like(wave_obs, dtype=float)                                                                                                                                 
                                                                                                                                                                        
    # complete absorption below Lyman limit (912 Å rest-frame)                                                                                                              
    T[wave_emit < 912.0] = 0.0
                                                                                                                                                                            
    # Lyman-alpha forest: 912 < lambda_rest < 1216 Å                                                                                                                        
    mask = (wave_emit >= 912.0) & (wave_emit < 1216.0)
    if np.any(mask):                                                                                                                                                        
        # effective optical depth: tau = 0.0036 * (lambda_obs / 1216)^3.46                                                                                                
        tau = 0.0036 * (wave_obs[mask] / 1216.0) ** 3.46                                                                                                                    
        T[mask] = np.exp(-tau)
                                                                                                                                                                            
    return T                                                                                                                                                              
                                                                                                                                                                            
                                                                                                                                                                        
# apply to z=2.5 simulation
z = 2.5
params['redshift'] = z
freq = frequencies[0]  # e.g. g-band                                                                                                                                        

flux = extinction_with_tde_base_model(                                                                                                                                      
    times, **params,                                                                                                                                                      
    frequency=np.full(len(times), freq),                                                                                                                                    
    output_format='flux_density',                                                                                                                                         
    base_model='tde_analytical',
)                                                                                                                                                                           

T = igm_transmission(np.array([freq]), z)[0]  # scalar transmission for this band                                                                                           
flux_igm = flux * T                                                                                                                                                       
                                                                                                                                                                            
fig, ax = plt.subplots(figsize=(10, 5))                                                                                                                                     
ax.plot(times, flux, label=f'z={z} no IGM')
ax.plot(times, flux_igm, label=f'z={z} + IGM (Madau 1995)', linestyle='--')                                                                                                 
ax.set_xlabel('Observer-frame time (days)')                                                                                                                                 
ax.set_ylabel('Flux density (mJy)')
ax.set_title(f'g-band transmission at z={z}: T={T:.3f}')                                                                                                                    
ax.legend()                                                                                                                                                                 
plt.show()                     

# Get LSST sampled light curves

### [LSST pointing db](https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/baseline/)

In [ ]:
import importlib
import thor.utils.redback_sims
importlib.reload(thor.utils.redback_sims)

from thor.utils.redback_sims import load_redback_result
from thor.utils.redback_sims import RubinSimulator

In [ ]:
# load the TDE

# --- configure these ---
name = 'ZTF18acecugr'
model='tde_analytical'
redshift = None   # None → use fitted redshift from posterior; float → override
n_sims   = 1
tmax_days = 360 # none will plot the full 10 years of survey pointings
survey = "baseline" # options are "baseline" or "ddf" to plot deep drilling field observations
# -----------------------

result, transient = load_redback_result(
    name=name,
    data_source='fink_tde',
    model=model,
    nlive=500,
    exclude_bands=['W2', 'M2', 'W1', 'U'],
    base_dir='../../data/redback_sims/redback_inference',
)

In [ ]:
# load the TDE simulated with hand set parameters and tde_analytical model

# --- configure these ---
name = 'ZTF18acecugr'
model='tde_analytical'
redshift = None   # None → use fitted redshift from posterior; float → override
n_sims   = 1
tmax_days = 360 # none will plot the full 10 years of survey pointings
survey = "baseline" # options are "baseline" or "ddf" to plot deep drilling field observations
# -----------------------

result, transient = load_redback_result(
    name=name,
    data_source='fink_tde',
    model=model,
    nlive=500,
    exclude_bands=['W2', 'M2', 'W1', 'U'],
    base_dir='../../data/redback_sims/redback_inference',
)

In [ ]:
# load the TDE simulated with hand set parameters and tde_fallback model

# --- configure these ---
name = 'ZTF18acecugr'
model='tde_analytical'
redshift = None   # None → use fitted redshift from posterior; float → override
n_sims   = 1
tmax_days = 360 # none will plot the full 10 years of survey pointings
survey = "baseline" # options are "baseline" or "ddf" to plot deep drilling field observations
# -----------------------

result, transient = load_redback_result(
    name=name,
    data_source='fink_tde',
    model=model,
    nlive=500,
    exclude_bands=['W2', 'M2', 'W1', 'U'],
    base_dir='../../data/redback_sims/redback_inference',
)

In [ ]:
# load the SNIa example

# --- configure these ---
name = 'ZTF17aabvong'
model='salt2'
redshift = None   # None → use fitted redshift from posterior; float → override
n_sims   = 1
tmax_days = 360 # none will plot the full 10 years of survey pointings
survey = "baseline" # options are "baseline" or "ddf" to plot deep drilling field observations
# -----------------------

result, transient = load_redback_result(
    name=name,
    data_source='fink_sn',
    model=model,
    nlive=500,
    exclude_bands=['W2', 'M2', 'W1', 'U'],
    base_dir='../../data/redback_sims/redback_inference',
)

In [ ]:
sim = RubinSimulator(
    result=result,
    model=model,
    survey=survey,
    data_dir='../../data',
    redshift=redshift,
)

lightcurves = sim.simulate(n_sims=1)
sim.plot(tmax_days=360, name=name)

In [ ]:
# simulating from hand set params for a given model
# simulate for tde_analytical model

# --- configure these ---
model='tde_analytical'
redshift = 0.5   
n_sims   = 1
tmax_days = 360 # none will plot the full 10 years of survey pointings
survey = "ddf" # options are "baseline" or "ddf" to plot deep drilling field observations
# -----------------------

sim = RubinSimulator.from_params(                                                                                                                                         
    params={                                                                                                                                                                
        'mej': 1.0,                                                                                                                                                         
        'vej': 1e4,                                                                                                                                                         
        'kappa': 0.1,                                                                                                                                                       
        'kappa_gamma': 10.0,                                                                                                                                                
        'temperature_floor': 5000.0,
        'l0': 1e55,                                                                                                                                                         
        't_0_turn': 50.0,
        'redshift': redshift,                                                                                                                                                    
    },                                                                                                                                                                    
    model=model,
    survey=survey,                                                                                                                                                      
    data_dir='../../data',
    filters=['g', 'r', 'i', 'z'],                                                                                                                                           
)                                                                                                                                                                         
sim.simulate()                              
sim.plot(tmax_days=360, name='TDE from tde_analytical model')  

In [ ]:
# simulate for tde_fallback (mosfit) model

# --- configure these ---
model='tde_fallback'
redshift = 0.1   
n_sims   = 1
tmax_days = 360 # none will plot the full 10 years of survey pointings
survey = "ddf" # options are "baseline" or "ddf" to plot deep drilling field observations
# -----------------------

sim = RubinSimulator.from_params(                                                                                                                                         
    params = {
        'mbh6': 1.0,      # BH mass in units of 10^6 solar masses
        'mstar': 1.0,     # disrupted star mass (solar masses)
        'tvisc': 10.0,    # viscous timescale (days)
        'bb': 1.0,        # beta/gamma parameter for disrupted star
        'eta': 0.1,       # BH accretion efficiency
        'leddlimit': 1.0, # Eddington limit factor
        'rph0': 1.0,      # initial photosphere radius
        'lphoto': 1.0,    # photosphere luminosity factor
        'redshift': redshift
    },                                                                                                                                                                   
    model=model,
    survey=survey,                                                                                                                                                      
    data_dir='../../data',
    filters=['g', 'r', 'i', 'z'],                                                                                                                                           
)                                                                                                                                                                         
sim.simulate()                              
sim.plot(tmax_days=360, name='TDE from tde_fallback model')  